# IE-OPT-09 · 巡检回路（MTZ 公式）

**场景**：工程师从澄口调度室出发，要到柳塘泵站、青岗变电所、沙湾备件库和枫桥冷却塔各检查一次，最后回到调度室。路程按给定的车程分钟数计算，求总时间最短的巡回路线。

五个点的对称车程是合成教学数据。模型写成有向回路：两个方向都允许走，费用矩阵是对称的，所以正反两条路费用相同。

练习用，不计正式成绩。

## 1. 模型要点

- $x_{ij}$ 为 0-1，表示路线是否从 $i$ 直接到 $j$。每个点恰好离开一次、进入一次。
- 只靠进出各一次，仍可能拆成几个互不相连的小圈。
- MTZ 用连续变量 $u_i$（$i$ 不是起点）记录该点在巡回路上的顺位，$1 \le u_i \le n-1$。
- 对起点以外的两点：$u_i - u_j + n x_{ij} \le n - 1$。一旦选择 $i \to j$，$u_j$ 至少比 $u_i$ 大 1，不经过起点的小圈无法闭合。

这里没有时间窗，也没有「某些点必须先于另一些点」的先后要求。

In [ ]:
from itertools import permutations
from pyscipopt import Model, quicksum

places = ["澄口调度室", "柳塘泵站", "青岗变电所", "沙湾备件库", "枫桥冷却塔"]
travel = [
    [0, 12, 18, 15, 21],
    [12, 0, 14, 23, 16],
    [18, 14, 0, 11, 19],
    [15, 23, 11, 0, 13],
    [21, 16, 19, 13, 0],
]
assert all(travel[i][j] == travel[j][i] for i in range(5) for j in range(5))
assert all(travel[i][i] == 0 for i in range(5))

## 2. 建立 MTZ 模型

In [ ]:
def solve_tsp(travel):
    n = len(travel)
    model = Model("IE-OPT-09 tsp mtz")
    model.hideOutput(True)
    x = {(i, j): model.addVar(vtype="B", name=f"go_{i}_{j}")
         for i in range(n) for j in range(n) if i != j}
    u = {i: model.addVar(lb=1, ub=n - 1, vtype="C", name=f"order_{i}") for i in range(1, n)}
    model.setObjective(quicksum(travel[i][j] * x[i, j] for i, j in x), "minimize")
    for i in range(n):
        model.addCons(quicksum(x[i, j] for j in range(n) if j != i) == 1)
        model.addCons(quicksum(x[j, i] for j in range(n) if j != i) == 1)
    for i in range(1, n):
        for j in range(1, n):
            if i != j:
                model.addCons(u[i] - u[j] + n * x[i, j] <= n - 1)
    model.optimize()
    status = str(model.getStatus())
    if status != "optimal":
        raise RuntimeError(f"Solver status is {status}; optimality not certified")
    nxt = {i: next(j for j in range(n) if j != i and model.getVal(x[i, j]) > 0.5)
           for i in range(n)}
    tour = [0]
    while len(tour) <= n:
        tour.append(nxt[tour[-1]])
        if tour[-1] == 0:
            break
    return {"objective": float(model.getObjVal()), "tour": tour}

result = solve_tsp(travel)
print("最短总车程:", round(result["objective"]))
print("巡检顺序:", " → ".join(places[i] for i in result["tour"]))

## 3. 枚举起点之后的访问顺序

五个点去掉起点后只有 $4! = 24$ 种顺序。每条顺序加上从调度室出发和返回，就是一条完整巡回路线。

In [ ]:
def oracle_tour_cost(travel):
    n = len(travel)
    best = float("inf")
    best_order = None
    for order in permutations(range(1, n)):
        route = (0,) + order + (0,)
        total = sum(travel[route[i]][route[i + 1]] for i in range(n))
        if total < best:
            best, best_order = total, route
    return best, best_order

oracle, oracle_route = oracle_tour_cost(travel)
print("独立枚举最短车程:", oracle)
print("一条最优顺序:", oracle_route)
assert abs(result["objective"] - oracle) < 1e-6
assert result["tour"][0] == 0 and result["tour"][-1] == 0
assert sorted(result["tour"][:-1]) == list(range(len(places)))
print("通过：求解器目标与 24 种顺序的最小值一致，且每个点恰好经过一次")

## 4. 可以接着改什么

1. 把某一段车程改成严重绕路，看顺序是否避开它。
2. 想加「每个点有服务时间和到达时间窗」时，这个 MTZ 不够，那是 VRPTW，本轮不展开。
3. 点再多一些时，24 的阶乘会很快不能枚举。枚举只用来核对这个五节点例子，不是通用解法。